# Regression pour maintenance predictive

Objectif : rendre la regression plus interessante en ajoutant des flags metier de degradation, puis predire un RUL (`Remaining Useful Life`) plus coherent.

Le notebook ne modifie pas les CSV propres existants. Il genere un dataset enrichi dans `artifacts/regression_dataset_with_flags.csv`.

In [18]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = "data/clean/scada_capteurs_propre.csv"
RAW_DATA_PATH = "data/raw/scada_capteurs_sale.csv"
ARTIFACT_DIR = "artifacts"

os.makedirs(ARTIFACT_DIR, exist_ok=True)
pd.set_option("display.max_columns", 80)

## 1. Chargement des donnees

On repart du fichier SCADA propre. Si les timestamps propres sont mal reconvertis en 1970, on reprend les timestamps bruts pour calculer les dates de maintenance.

In [19]:
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

if df["timestamp"].dt.year.median() < 2000 and os.path.exists(RAW_DATA_PATH):
    raw_timestamps = pd.read_csv(RAW_DATA_PATH, usecols=["timestamp"])
    df["timestamp"] = pd.to_datetime(raw_timestamps["timestamp"], errors="coerce").fillna(df["timestamp"])

print(df.shape)
df.head()

(10000, 33)


,timestamp,machine_id,cycle_time_sec,Temperature_C,Vibration_mms,Sound_dB,Oil_Level_pct,Coolant_Level_pct,Hydraulic_Pressure_bar,Coolant_Flow_L_min,Heat_Index,Power_Consumption_kW,Operational_Hours,Error_Codes_Last_30_Days,sensor_anomaly_score,AI_Override_Events,flag_temperature_high,flag_vibration_high,flag_sound_high,flag_oil_low,flag_coolant_low,flag_pressure_low,flag_coolant_flow_low,flag_heat_high,flag_power_high,flag_error_codes_high,flag_anomaly_high,degradation_score,degradation_rate_pct,predicted_failure_probability,Remaining_Useful_Life_days,Failure_Within_7_Days,Maintenance_Required_Within_45_Days
0,2026-01-01 06:00:00,M04,49.60,64.79,2.69,70.120,83.09,69.08,131.660,22.95,75.99,44.57,1582.08,5.0,0.373,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.8,18.4,0.366,203.0,0,0.0
1,2026-01-01 06:05:00,M04,47.48,71.64,2.70,73.855,83.09,66.31,124.175,25.31,73.48,50.60,1581.73,3.0,0.372,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.4,9.2,0.299,203.0,0,0.0
2,2026-01-01 06:10:00,M03,49.17,67.48,1.89,75.560,76.56,73.56,128.270,33.15,65.67,46.84,1150.17,0.0,0.271,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.144,263.9,0,0.0
3,2026-01-01 06:15:00,M03,44.15,67.48,2.08,69.590,83.45,80.59,126.900,31.00,58.34,40.18,1133.33,3.0,0.203,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.035,284.0,0,0.0
4,2026-01-01 06:20:00,M01,43.28,66.33,1.90,75.630,82.90,88.83,124.900,35.39,68.91,45.25,945.33,1.0,0.424,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.171,229.3,0,0.0


## 2. Creation des flags de degradation

Chaque flag represente un signal industriel anormal : vibration elevee, temperature elevee, huile basse, liquide de refroidissement bas, pression basse, etc.

Ces flags rendent le probleme plus lisible : le modele apprend a relier des symptomes de degradation a un RUL plus faible.

In [20]:
df_flags = df.copy()

df_flags["flag_temperature_high"] = (df_flags["Temperature_C"] > 68).astype(int)
df_flags["flag_vibration_high"] = (df_flags["Vibration_mms"] > 2.6).astype(int)
df_flags["flag_sound_high"] = (df_flags["Sound_dB"] > 82).astype(int)
df_flags["flag_oil_low"] = (df_flags["Oil_Level_pct"] < 78).astype(int)
df_flags["flag_coolant_low"] = (df_flags["Coolant_Level_pct"] < 72).astype(int)
df_flags["flag_pressure_low"] = (df_flags["Hydraulic_Pressure_bar"] < 115).astype(int)
df_flags["flag_coolant_flow_low"] = (df_flags["Coolant_Flow_L_min"] < 25).astype(int)
df_flags["flag_heat_high"] = (df_flags["Heat_Index"] > 75).astype(int)
df_flags["flag_power_high"] = (df_flags["Power_Consumption_kW"] > 55).astype(int)
df_flags["flag_error_codes_high"] = (df_flags["Error_Codes_Last_30_Days"] >= 3).astype(int)
df_flags["flag_anomaly_high"] = (df_flags["sensor_anomaly_score"] >= 0.70).astype(int)
df_flags["flag_operational_hours_high"] = (df_flags["Operational_Hours"] >= 1200).astype(int)

flag_columns = [col for col in df_flags.columns if col.startswith("flag_")]
df_flags[flag_columns].mean().sort_values(ascending=False)

flag_operational_hours_high    0.8305
flag_vibration_high            0.6213
flag_error_codes_high          0.6049
flag_temperature_high          0.5647
flag_oil_low                   0.5647
flag_coolant_low               0.3684
flag_heat_high                 0.3348
flag_pressure_low              0.2747
flag_power_high                0.1427
flag_coolant_flow_low          0.1410
flag_anomaly_high              0.0753
flag_sound_high                0.0538
dtype: float64

## 3. Construction d'un taux de degradation metier

On pondere les flags selon leur importance. Par exemple, les vibrations, la temperature, l'huile basse et les erreurs recentes pesent plus lourd.

Le taux obtenu est compris entre 0 et 100%.

In [21]:
weights = {
    "flag_vibration_high": 2.0,
    "flag_temperature_high": 1.6,
    "flag_oil_low": 1.5,
    "flag_coolant_low": 1.4,
    "flag_pressure_low": 1.4,
    "flag_error_codes_high": 1.8,
    "flag_anomaly_high": 1.5,
    "flag_heat_high": 1.2,
    "flag_power_high": 1.0,
    "flag_sound_high": 0.8,
    "flag_coolant_flow_low": 1.0,
    "flag_operational_hours_high": 1.0,
}

weighted_score = sum(df_flags[col] * weight for col, weight in weights.items())
max_score = sum(weights.values())

df_flags["degradation_score"] = weighted_score.round(2)
df_flags["degradation_rate_pct"] = (weighted_score / max_score * 100).clip(0, 100).round(1)

df_flags[["machine_id", "degradation_score", "degradation_rate_pct"] + flag_columns].head()

,machine_id,degradation_score,degradation_rate_pct,flag_temperature_high,flag_vibration_high,flag_sound_high,flag_oil_low,flag_coolant_low,flag_pressure_low,flag_coolant_flow_low,flag_heat_high,flag_power_high,flag_error_codes_high,flag_anomaly_high,flag_operational_hours_high
0,M04,8.4,51.9,0,1,0,0,1,0,1,1,0,1,0,1
1,M04,7.8,48.1,1,1,0,0,1,0,0,0,0,1,0,1
2,M03,1.5,9.3,0,0,0,1,0,0,0,0,0,0,0,0
3,M03,1.8,11.1,0,0,0,0,0,0,0,0,0,1,0,0
4,M01,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0


## 4. Creation d'une cible RUL plus coherente

Dans le CSV initial, `Remaining_Useful_Life_days` est peu correle aux capteurs. Pour tester la regression, on cree donc une cible experimentale :

- plus la degradation est forte, plus le RUL baisse ;
- si une panne est indiquee dans les 7 jours, on force un RUL plus court ;
- on ajoute un bruit leger pour garder un comportement realiste.

In [22]:
rng = np.random.default_rng(42)

base_rul = 365 - (df_flags["degradation_rate_pct"] / 100 * 330)
wear_penalty = ((df_flags["Operational_Hours"] - df_flags["Operational_Hours"].min()) /
                (df_flags["Operational_Hours"].max() - df_flags["Operational_Hours"].min()) * 35)
noise = rng.normal(0, 8, len(df_flags))

df_flags["engineered_rul_days"] = (base_rul - wear_penalty + noise).clip(1, 365)

if "Failure_Within_7_Days" in df_flags.columns:
    failure_mask = df_flags["Failure_Within_7_Days"].astype(int) == 1
    df_flags.loc[failure_mask, "engineered_rul_days"] = df_flags.loc[failure_mask, "engineered_rul_days"].clip(1, 45)

df_flags["engineered_rul_days"] = df_flags["engineered_rul_days"].round(1)
df_flags["engineered_maintenance_date"] = df_flags["timestamp"] + pd.to_timedelta(
    np.ceil(df_flags["engineered_rul_days"]), unit="D"
)

df_flags[["machine_id", "timestamp", "degradation_rate_pct", "engineered_rul_days", "engineered_maintenance_date"]].head(10)

,machine_id,timestamp,degradation_rate_pct,engineered_rul_days,engineered_maintenance_date
0,M04,2026-01-01 06:00:00,51.9,175.9,2026-06-26 06:00:00
1,M04,2026-01-01 06:05:00,48.1,177.7,2026-06-28 06:05:00
2,M03,2026-01-01 06:10:00,9.3,333.1,2026-12-01 06:10:00
3,M03,2026-01-01 06:15:00,11.1,329.2,2026-11-27 06:15:00
4,M01,2026-01-01 06:20:00,0.0,348.4,2026-12-16 06:20:00
5,M04,2026-01-01 06:25:00,57.4,144.6,2026-05-26 06:25:00
6,M03,2026-01-01 06:30:00,11.1,322.2,2026-11-20 06:30:00
7,M02,2026-01-01 06:35:00,29.6,252.2,2026-09-11 06:35:00
8,M03,2026-01-01 06:40:00,23.5,279.9,2026-10-08 06:40:00
9,M02,2026-01-01 06:45:00,43.8,200.8,2026-07-21 06:45:00


## 5. Entrainement des modeles de regression

La cible predite devient `engineered_rul_days`.

On ajoute les flags aux features numeriques, car ils rendent les signaux de degradation explicites.

In [23]:
numeric_features = [
    "cycle_time_sec",
    "Temperature_C",
    "Vibration_mms",
    "Sound_dB",
    "Oil_Level_pct",
    "Coolant_Level_pct",
    "Hydraulic_Pressure_bar",
    "Coolant_Flow_L_min",
    "Heat_Index",
    "Power_Consumption_kW",
    "Operational_Hours",
    "Error_Codes_Last_30_Days",
    "sensor_anomaly_score",
]

features = numeric_features + flag_columns + ["degradation_score", "degradation_rate_pct"]
target = "engineered_rul_days"

X = df_flags[features]
y = df_flags[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "LinearRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ]),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=3,
    ),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42),
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        "model": name,
        "mae_days": mean_absolute_error(y_test, pred),
        "rmse_days": np.sqrt(mean_squared_error(y_test, pred)),
        "r2_score": r2_score(y_test, pred),
    })
    fitted_models[name] = model

results_df = pd.DataFrame(results).sort_values("mae_days")
results_df

,model,mae_days,rmse_days,r2_score
0,LinearRegression,6.765452,10.099890,0.987089
2,GradientBoostingRegressor,6.884532,10.177625,0.986890
1,RandomForestRegressor,7.123116,10.457302,0.986159


## 6. Predictions finales et fichiers generes

On garde le modele avec la plus faible MAE, puis on calcule les predictions de maintenance.

In [24]:
best_model_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]

df_flags["predicted_rul_days"] = np.clip(best_model.predict(X), 1, 365).round(1)
df_flags["predicted_degradation_rate_pct"] = ((1 - df_flags["predicted_rul_days"] / 365) * 100).clip(0, 100).round(1)
df_flags["recommended_maintenance_date"] = df_flags["timestamp"] + pd.to_timedelta(
    np.ceil(df_flags["predicted_rul_days"]), unit="D"
)
df_flags["maintenance_priority"] = pd.cut(
    df_flags["predicted_rul_days"],
    bins=[0, 7, 30, 90, 365],
    labels=["urgent", "high", "medium", "low"],
)

dataset_path = os.path.join(ARTIFACT_DIR, "regression_dataset_with_flags.csv")
comparison_path = os.path.join(ARTIFACT_DIR, "regression_with_flags_comparison.csv")
predictions_path = os.path.join(ARTIFACT_DIR, "maintenance_predictions_with_flags.csv")
model_path = os.path.join(ARTIFACT_DIR, "best_rul_regressor_with_flags.pkl")

df_flags.to_csv(dataset_path, index=False)
results_df.to_csv(comparison_path, index=False)
joblib.dump(best_model, model_path)

prediction_columns = [
    "timestamp",
    "machine_id",
    "predicted_rul_days",
    "predicted_degradation_rate_pct",
    "recommended_maintenance_date",
    "maintenance_priority",
    "degradation_rate_pct",
    "engineered_rul_days",
] + flag_columns

predictions = df_flags[prediction_columns].sort_values("predicted_rul_days")
predictions.to_csv(predictions_path, index=False)

print("Meilleur modele :", best_model_name)
print("Dataset enrichi :", dataset_path)
print("Comparaison modeles :", comparison_path)
print("Predictions :", predictions_path)
print("Modele :", model_path)

predictions.head(15)

Meilleur modele : LinearRegression
Dataset enrichi : artifacts\regression_dataset_with_flags.csv
Comparaison modeles : artifacts\regression_with_flags_comparison.csv
Predictions : artifacts\maintenance_predictions_with_flags.csv
Modele : artifacts\best_rul_regressor_with_flags.pkl


,timestamp,machine_id,predicted_rul_days,predicted_degradation_rate_pct,recommended_maintenance_date,maintenance_priority,degradation_rate_pct,engineered_rul_days,flag_temperature_high,flag_vibration_high,flag_sound_high,flag_oil_low,flag_coolant_low,flag_pressure_low,flag_coolant_flow_low,flag_heat_high,flag_power_high,flag_error_codes_high,flag_anomaly_high,flag_operational_hours_high
9985,2026-02-04 22:05:00,M04,1.0,99.7,2026-02-05 22:05:00,urgent,100.0,7.8,1,1,1,1,1,1,1,1,1,1,1,1
9996,2026-02-04 23:00:00,M04,1.0,99.7,2026-02-05 23:00:00,urgent,100.0,1.0,1,1,1,1,1,1,1,1,1,1,1,1
8839,2026-01-31 22:35:00,M04,1.0,99.7,2026-02-01 22:35:00,urgent,100.0,6.5,1,1,1,1,1,1,1,1,1,1,1,1
9920,2026-02-04 16:40:00,M04,1.0,99.7,2026-02-05 16:40:00,urgent,100.0,15.5,1,1,1,1,1,1,1,1,1,1,1,1
8243,2026-01-29 20:55:00,M04,1.0,99.7,2026-01-30 20:55:00,urgent,100.0,14.5,1,1,1,1,1,1,1,1,1,1,1,1
9740,2026-02-04 01:40:00,M04,1.0,99.7,2026-02-05 01:40:00,urgent,100.0,15.3,1,1,1,1,1,1,1,1,1,1,1,1
8814,2026-01-31 20:30:00,M04,1.0,99.7,2026-02-01 20:30:00,urgent,100.0,15.9,1,1,1,1,1,1,1,1,1,1,1,1
9571,2026-02-03 11:35:00,M04,1.0,99.7,2026-02-04 11:35:00,urgent,100.0,1.0,1,1,1,1,1,1,1,1,1,1,1,1
9951,2026-02-04 19:15:00,M04,1.0,99.7,2026-02-05 19:15:00,urgent,100.0,8.3,1,1,1,1,1,1,1,1,1,1,1,1
8991,2026-02-01 11:15:00,M04,1.0,99.7,2026-02-02 11:15:00,urgent,100.0,1.0,1,1,1,1,1,1,1,1,1,1,1,1


## Interpretation attendue

Avec les flags, le modele doit mieux apprendre car la cible RUL est construite a partir d'une logique industrielle explicite.

A presenter ainsi :

> Nous avons enrichi les donnees capteurs avec des flags de degradation, puis construit une cible RUL coherente avec ces signaux. Le modele de regression estime le nombre de jours restants avant maintenance. Ce RUL permet ensuite de calculer un taux de degradation et une date recommandee d'intervention.

Attention : cette cible est experimentale/simulee. Dans un cas industriel reel, elle doit etre remplacee par des historiques de pannes et de maintenances reels.